<a href="https://colab.research.google.com/github/rohanyashraj/iai-workshop/blob/main/01_genai_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · GenAI Basics — your first calls to the reasoner

**Agentic AI for Health Actuaries** · IAI Seminar · 25 August 2026 · Hub: `github.com/rohanyashraj/iai-workshop`

> All data in this notebook is **hypothetical** — ABC Health is a fictional entity calibrated to plausible Indian health insurance experience, for teaching only.

**Used in:** Session 1, Part 2 (The Reasoner).
**You will:** make your first Gemini API call, practise the CCCE prompt discipline, watch a hallucination happen on demand, and get guaranteed-parseable JSON out of an LLM.

**Setup (2 minutes):**
1. Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
2. In Colab, click the **key icon** (left sidebar) → *Add new secret* → name it `GOOGLE_API_KEY`, paste the key, toggle notebook access ON.
3. Run the cells top to bottom (`Runtime → Run all` after setup).

In [1]:
%pip install -q "google-genai==2.19.0"

/Users/rohanyashraj/Documents/Actuarial/2026 IFoA Workshop/25th August 2026/iai-workshop/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


## §1 · Auth — the key never appears in the notebook
Colab Secrets keeps the key out of the notebook file. This is the same hygiene you will use for every agent you ship: secrets live in a store, never in code.

In [2]:
import os
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("GOOGLE_API_KEY loaded from Colab Secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
    print("Not running in Colab (or no secret set) - falling back to a local .env / environment variable.")
from google import genai
from IPython.display import Markdown, display

client = genai.Client()
MODEL = "gemini-3.5-flash-lite"   # PINNED — silent model drift is an audit failure
print("Client ready, model pinned to:", MODEL)

Not running in Colab (or no secret set) - falling back to a local .env / environment variable.


Client ready, model pinned to: gemini-3.5-flash-lite


## §2 · First call — define IBNR for a board member

In [3]:
response = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a non-actuarial board member, in one line.",
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n--- usage ---")
print(response.usage_metadata)   # token counts: you will care about these when agents multiply call volume

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


📋 GEMINI MODEL RESPONSE


"IBNR is the money we estimate we will eventually have to pay for claims that have already happened, but haven’t been reported to us yet."


END OF MODEL RESPONSE

--- usage ---
cache_tokens_details=None cached_content_token_count=None candidates_token_count=31 candidates_tokens_details=None prompt_token_count=19 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=19
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=50 traffic_type=None


## §3 · CCCE — Clarity, Context, Constraints, Examples
The prompt below is the worked example from the slides: an IBNR commentary for ABC Health Q3 2024. Each bracketed fragment does exactly one job — edit any part without breaking the others.

**Exercise:** delete the Constraints block, re-run, and compare. Then rewrite the prompt for *your* line of business.

In [4]:
ccce_prompt = """
[Clarity] Write a two-paragraph commentary on the IBNR result for ABC Health Q3 2024.
[Context] Indemnity health book. Chain-ladder ultimate INR 186 Cr vs prior estimate INR 172 Cr.
Q3 saw a hospital network strike in two states.
[Constraints] Audience: appointed actuary peer-review meeting. Max 180 words.
Do not invent figures. Cite only the figures provided above.
[Example] Voice to match: "The Q2 ultimate of INR 164 Cr increased to INR 172 Cr after the network
expansion in Tier 2 cities..."
"""
resp = client.models.generate_content(model=MODEL, contents=ccce_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


The Q3 ultimate IBNR for the indemnity health book increased to INR 186 Cr from the prior estimate of INR 172 Cr. This upward movement was driven by operational disruptions stemming from the hospital network strike in two states during the quarter. 

The strike caused acute reporting delays and temporarily compressed paid-to-date development patterns. Consequently, standard chain-ladder projections required higher reserve recognition to account for the delayed emergence of claims.


END OF MODEL RESPONSE


### §3.1 · Demo 1 — the vague version, for contrast
Run the deliberately vague prompt below, then re-run the CCCE version above and **diff the outputs**. Same model, same cost — the entire quality delta is the prompt.

In [5]:
vague = "Write about IBNR for our board."
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=vague).text[:800]))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Expect: a generic essay that INVENTS plausible numbers (we gave it none)
# and lands in a register somewhere between textbook and LinkedIn.

📋 GEMINI MODEL RESPONSE


Here is a memorandum tailored for a Board of Directors. It balances high-level strategic oversight with necessary technical context, avoiding overly dense actuarial jargon while emphasizing financial governance and risk management.

***

**MEMORANDUM**

**TO:** Board of Directors  
**FROM:** [Your Name/Title, e.g., Chief Financial Officer / Chief Risk Officer]  
**DATE:** October 24, 2023  
**SUBJECT:** Understanding and Overseeing IBNR (Incurred But Not Reported) Reserves  

---

### 1. Executive Summary
For an insurance or risk-bearing entity, the accuracy of our financial statements hinges on our reserves. Among these, **IBNR (Incurred But Not Reported)** represents our largest and most complex liability. 

In short, IBNR is an estimate of the money we need to set aside today to pay for


END OF MODEL RESPONSE


### §3.2 · Demo 2 — one fact, two audiences
Audience is a prompt parameter. Same reserve-strengthening fact, rendered for a board member and for a new student. Note: the model *dresses* the fact we supply — it does not source it.

In [6]:
fact = ("We strengthened PMI hospitalisation reserves by INR 42 Cr "
        "following a sharp rise in empanelled-hospital tariffs.")

for audience, style in [
    ("board member", "2 sentences, business impact first, no jargon"),
    ("new actuarial student", "4 sentences, explain WHY tariffs drive PMI reserves, define terms"),
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Explain: {fact} For a {audience}. {style}")
    # Clear visual separation
    print("=" * 70)
    print("📋 GEMINI MODEL RESPONSE")
    print("=" * 70)

    display(Markdown(f"**{audience.upper()}**\n\n{r.text}"))

    print("\n" + "=" * 70)
    print("END OF MODEL RESPONSE")
    print("=" * 70)


📋 GEMINI MODEL RESPONSE


**BOARD MEMBER**

We set aside an additional 42 crore rupees to cover higher-than-expected medical bills from our partner hospitals, directly protecting our future profitability. This financial adjustment ensures our reserves accurately reflect the surging cost of healthcare.


END OF MODEL RESPONSE


📋 GEMINI MODEL RESPONSE


**NEW ACTUARIAL STUDENT**

PMI (Private Medical Insurance) reserves are pools of capital insurers set aside to pay future health insurance claims. When empanelled-hospital tariffs—the agreed-upon prices hospitals charge for treatments—rise sharply, the cost of medical claims directly increases. Because these higher costs outpace previous pricing assumptions, actuaries must strengthen (increase) reserves to ensure the company remains solvent and can cover future payouts. In short, inflating medical costs force insurers to hold more money today to cover the escalating price of tomorrow's claims.


END OF MODEL RESPONSE


### §3.3 · Demo 3 — few-shot examples tame formatting
Show, don't tell: two worked examples buy you the delimiter, the casing, the arrow convention, and no chatty preamble. **Exercise:** feed it a genuinely weird input and see whether the pattern holds.

In [7]:
prompt = """Convert each change to the format of the examples.

EXAMPLES
In: Hospitalisation frequency moved from 3.2% to 3.5% for ages 45+.
Out: HOSP_FREQ | age 45+ | 3.2% -> 3.5%
In: Claim severity trend up 40bps.
Out: SEV_TREND | all | +40bps

NOW CONVERT
In: CI incidence for cardiac conditions, ages 40-55, moves from 0.45% to 0.52%.
Out:"""
print(client.models.generate_content(model=MODEL, contents=prompt).text)


Out: CI_INCIDENCE | age 40-55 | 0.45% -> 0.52%


### §3.4 · Demo 4 — step-by-step reasoning (with a warning label)
Asking for steps improves reliability — it does **not** guarantee it. Re-run this cell three times: do the running totals stay identical? This is why the afternoon's agent does arithmetic in *Python* and lets Gemini narrate.

In [8]:
prompt = """A PMI policy has base premium INR 9,000 with relativities:
age band 46-55 = 1.45, sum insured 10L = 1.30, family floater = 1.10, NCB 30% = 0.70.
Walk through the premium calculation STEP BY STEP, showing the running
total after each factor, then state the final premium."""
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=prompt).text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Check by hand: 9000 * 1.45 * 1.30 * 1.10 * 0.70 = 13,063.05


📋 GEMINI MODEL RESPONSE


Here is the step-by-step walk-through for calculating the Private Medical Insurance (PMI) premium:

* **Base Premium:** INR 9,000

---

### **Step 1: Apply Age Band Relativity (46–55)**
* **Relativity:** 1.45
* **Calculation:** $\text{INR } 9,000 \times 1.45 = \text{INR } 13,050$
* **Running Total:** **INR 13,050**

---

### **Step 2: Apply Sum Insured Relativity (10L)**
* **Relativity:** 1.30
* **Calculation:** $\text{INR } 13,050 \times 1.30 = \text{INR } 16,965$
* **Running Total:** **INR 16,965**

---

### **Step 3: Apply Policy Type Relativity (Family Floater)**
* **Relativity:** 1.10
* **Calculation:** $\text{INR } 16,965 \times 1.10 = \text{INR } 18,661.50$
* **Running Total:** **INR 18,661.50**

---

### **Step 4: Apply No Claim Bonus (NCB 30%)**
* **Relativity:** 0.70 *(representing a 30% discount, i.e., 1 - 0.30)*
* **Calculation:** $\text{INR } 18,661.50 \times 0.70 = \text{INR } 13,063.05$
* **Running Total:** **INR 13,063.05**

---

### **Final Premium**
Rounding to the nearest paisa/rupee, the final calculated premium is **INR 13,063.05**.


END OF MODEL RESPONSE


### §3.5 · Demo review — the habit that IS the skill
1. CCCE moved quality more than a model upgrade would — specification beats horsepower.
2. Register control is leverage, but the model dresses facts; it doesn't source them.
3. Few-shot is a formatting contract — stress-test it before relying on it.
4. Step-by-step is transparency, not verified arithmetic.

**The loop:** prompt → output → review → edit prompt — the same loop you'll run on agent traces this afternoon.

## §4 · The hallucination demo — run it, believe it
We ask for a regulation that **does not exist**. The model will not say 'no such factor' — it will produce the most *plausible-sounding* answer, confidently.

⚠️ This exact failure mode reappears **inside your agent** in notebook 04 — and you will fix it with a guardrail tool.

In [9]:
hallucination_prompt = (
    "What is the IRDAI-mandated co-payment factor for senior-citizen PMI "
    "policies with sum insured above INR 10 lakh? "
    "Give the exact factor value and the section reference."
)
resp = client.models.generate_content(model=MODEL, contents=hallucination_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n⚠️  Verify before you trust: there is no such published factor. "
      "Whatever appears above was constructed to be plausible, not true.")


📋 GEMINI MODEL RESPONSE


Based on the Insurance Regulatory and Development Authority of India (IRDAI) Master Circular on Health Insurance Products (dated July 29, 2024), the details regarding the co-payment factor for senior-citizen health insurance policies are as follows:

*   **Co-payment Factor:** **Maximum of 30%** (Insurers may offer a lower co-payment, but it cannot exceed 30% for senior citizens for policies with a sum insured above INR 10 lakh, provided the policy is underwritten without medical underwriting or based on standard terms). 
*   **Section Reference:** **Clause 43 (ii)** of the *IRDAI (Health Insurance Products) Regulations, 2024* (as reiterated in the Master Circular on Health Insurance Products, Ref: IRDAI/HLT/REG/CIR/144/7/2024). 

*(Note: Under Clause 43(i) and (ii), while insurers can apply a co-payment up to a maximum of 30% for senior citizens aged 65 and above, insurers are strictly prohibited from imposing any mandatory co-payment based solely on entry age for policies below age 65).*


END OF MODEL RESPONSE

⚠️  Verify before you trust: there is no such published factor. Whatever appears above was constructed to be plausible, not true.


## §5 · Structured output — because agents speak JSON
One config line guarantees parseable JSON. This is how every component of an agentic system exchanges data — prose is only for humans at the last step.

In [10]:
import json

prompt = """For individual PMI (health) cover, list 5 rating factors.
For each: name, direction (increase/decrease premium), one-line justification. Return JSON."""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
factors = json.loads(resp.text)   # guaranteed to parse
for f in factors:
    print(f)


{'name': 'Age', 'direction': 'increase', 'justification': 'Older individuals have a statistically higher likelihood of requiring medical treatment and claiming on their policy.'}
{'name': 'Location', 'direction': 'increase', 'justification': 'Regions with higher costs of living and more expensive private healthcare facilities result in higher claim payouts.'}
{'name': 'Level of Excess', 'direction': 'decrease', 'justification': 'Agreeing to pay a higher voluntary excess shifts more initial financial risk to the policyholder, reducing insurer payouts.'}
{'name': 'Smoking Status', 'direction': 'increase', 'justification': 'Tobacco use significantly increases the risk of developing chronic health conditions and severe illnesses.'}
{'name': 'Hospital Network Tier', 'direction': 'decrease', 'justification': "Choosing a restricted or local hospital network rather than a comprehensive nationwide network lowers the insurer's facility costs."}


## §6 · Review exercise — mark the model's homework
Treat the JSON above as a junior analyst's first draft and grade it:

1. Is every **direction** consistent with your priors?
2. Did it name factors your book doesn't collect (e.g. a wellness-programme discount, occupation class)?
3. What material factors are **missing** (BMI band? room-rent category?)
4. What would you still need before any of this goes near a tariff filing? *(Hint: magnitudes → a GLM run → notebook 02.)*

**The rule that survives today:** the reasoner narrates; tools know; humans sign.

---
**Log what you ran.** For anything regulatory: save the full prompt–response pair, the model id, and the timestamp — 'the AI wrote it' is not a defence without the receipt.

In [11]:
# Minimal call log — one CSV row per call. In production this is your observability stack.
import datetime, csv, pathlib

def log_call(prompt, response_text, model=MODEL, path="genai_call_log.csv"):
    new = not pathlib.Path(path).exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["ts_utc", "model", "prompt", "response"])
        w.writerow([datetime.datetime.now(datetime.UTC).isoformat(), model, prompt, response_text])

log_call(prompt, resp.text)
print("logged — this habit is checklist question 10 in miniature")

logged — this habit is checklist question 10 in miniature
